In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
import math
from datetime import datetime, timedelta
from datetime import time

In [2]:
!rm -r "/content/results"

rm: cannot remove '/content/results': No such file or directory


In [3]:
import sys

# Add a custom folder to Python path
sys.path.append("/content/PatchTST")  # Now you can import modules from this folder



In [4]:
df = pd.read_csv('/content/RELIANCE_5minute.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

In [5]:
df.shape

(194552, 6)

In [6]:
end_date = pd.Timestamp('2025-08-22 15:15:00')  # specific end date
start_date = pd.Timestamp('2023-08-22 09:15:00')  # 3 years before

# Filter data within this range
df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]

# Extract time
df['time'] = df['date'].dt.time

# Market hours: 9:15 to 15:15
market_start = time(9, 15)
market_end = time(15, 15)

df= df[(df['time'] >= market_start) & (df['time'] <= market_end)]

In [7]:
df = df[["date", "close"]]

In [8]:
df.shape

(35374, 2)

In [9]:
!git clone https://github.com/ShubhamS1101/PatchTST.git

# upload dataset/my_stock.csv


Cloning into 'PatchTST'...
remote: Enumerating objects: 209, done.
remote: Counting objects: 100% (209/209), done.
remote: Compressing objects: 100% (200/200), done.
remote: Total 209 (delta 79), reused 34 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (209/209), 2.56 MiB | 19.27 MiB/s, done.
Resolving deltas: 100% (79/79), done.


In [10]:
df.to_csv("/content/PatchTST/dataset/stock.csv", index=False)

In [11]:
!pip install vmdpy


In [12]:
df_pred = df.head(200)
df_pred.to_csv("/content/PatchTST/dataset/stock_pred.csv", index=False)

In [13]:

import re

file_path = "/content/PatchTST/PatchTST/utils/tools.py"

# Read the content of the file
with open(file_path, 'r') as file:
    file_content = file.read()

# Replace np.Inf with np.inf
modified_content = re.sub(r'np\.Inf', 'np.inf', file_content)


# Write the modified content back to the file
with open(file_path, 'w') as file:
    file.write(modified_content)

print(f"Modified {file_path} to replace np.Inf with np.inf")

Modified /content/PatchTST/PatchTST/utils/tools.py to replace np.Inf with np.inf


In [ ]:
#!/bin/bash
# train_patchtst.sh
# Shell script to train PatchTST + VMD + AWSL model

# -----------------------
# Configuration
# -----------------------
ROOT_PATH="PatchTST/dataset/"
DATA_PATH="stock.csv"
FEATURES="S"
TARGET="close"
FREQ="h"
CHECKPOINTS="PatchTST/checkpoints/"

SEQ_LEN=96
LABEL_LEN=48
PRED_LEN=1

EMBED="timeF"
NUM_WORKERS=0

MODEL="PatchTST"
ENC_IN=1
E_LAYERS=3
N_HEADS=16
D_MODEL=128
D_FF=256
DROPOUT=0.1
FC_DROPOUT=0.1
HEAD_DROPOUT=0.0
PATCH_LEN=16
STRIDE=8
PADDING_PATCH="end"

DEVICE_IDS="0"
USE_GPU=True
USE_MULTI_GPU=False
TRAIN_EPOCHS=3
BATCH_SIZE=32
LEARNING_RATE=0.0001
PATIENCE=5
PCT_START=0.3
LRADJ="TST"

DECOMPOSITION=False
KERNEL_SIZE=25

USE_VMD=True
NUM_IMFS=3
USE_ASWL=False

DO_TRAIN=True
DO_TEST=False
DO_PREDICT=False

ITR=1
DES="experiment"

# -----------------------
# Run training
# -----------------------
!python3 PatchTST/run.py \
    --root_path $ROOT_PATH \
    --data_path $DATA_PATH \
    --features $FEATURES \
    --individual 0 \
    --target $TARGET \
    --freq $FREQ \
    --checkpoints $CHECKPOINTS \
    --seq_len $SEQ_LEN \
    --label_len $LABEL_LEN \
    --pred_len $PRED_LEN \
    --embed $EMBED \
    --num_workers $NUM_WORKERS \
    --model $MODEL \
    --enc_in $ENC_IN \
    --e_layers $E_LAYERS \
    --n_heads $N_HEADS \
    --d_model $D_MODEL \
    --d_ff $D_FF \
    --revin 1 \
    --affine 0 \
    --subtract_last 0\
    --dropout $DROPOUT \
    --fc_dropout $FC_DROPOUT \
    --head_dropout $HEAD_DROPOUT \
    --patch_len $PATCH_LEN \
    --stride $STRIDE \
    --padding_patch $PADDING_PATCH \
    --device_ids $DEVICE_IDS \
    --use_gpu $USE_GPU \
    $( [ "$USE_MULTI_GPU" = True ] && echo "--use_multi_gpu" ) \
    --train_epochs $TRAIN_EPOCHS \
    --batch_size $BATCH_SIZE \
    --learning_rate $LEARNING_RATE \
    --patience $PATIENCE \
    --pct_start $PCT_START \
    --lradj $LRADJ \
    $( [ "$DECOMPOSITION" = True ] && echo "--decomposition" ) \
    --kernel_size $KERNEL_SIZE \
    $( [ "$USE_VMD" = True ] && echo "--use_vmd" ) \
    --num_imfs $NUM_IMFS \
    $( [ "$USE_ASWL" = True ] && echo "--use_aswl" ) \
    $( [ "$DO_TRAIN" = True ] && echo "--do_train" ) \
    $( [ "$DO_TEST" = True ] && echo "--do_test" ) \
    $( [ "$DO_PREDICT" = True ] && echo "--do_predict" ) \
    --itr $ITR \
    --des $DES


2025-10-05 17:36:14,314 INFO Arguments:
2025-10-05 17:36:14,314 INFO   affine: 0
2025-10-05 17:36:14,314 INFO   batch_size: 32
2025-10-05 17:36:14,314 INFO   checkpoints: PatchTST/checkpoints/
2025-10-05 17:36:14,314 INFO   d_ff: 256
2025-10-05 17:36:14,314 INFO   d_model: 128
2025-10-05 17:36:14,314 INFO   data_path: stock.csv
2025-10-05 17:36:14,314 INFO   decomposition: False
2025-10-05 17:36:14,314 INFO   des: experiment
2025-10-05 17:36:14,314 INFO   device_ids: 0
2025-10-05 17:36:14,314 INFO   device_ids_list: [0]
2025-10-05 17:36:14,314 INFO   do_predict: False
2025-10-05 17:36:14,315 INFO   do_test: False
2025-10-05 17:36:14,315 INFO   do_train: True
2025-10-05 17:36:14,315 INFO   dropout: 0.1
2025-10-05 17:36:14,315 INFO   e_layers: 3
2025-10-05 17:36:14,315 INFO   embed: timeF
2025-10-05 17:36:14,315 INFO   enc_in: 1
2025-10-05 17:36:14,315 INFO   fc_dropout: 0.1
2025-10-05 17:36:14,315 INFO   features: S
2025-10-05 17:36:14,315 INFO   freq: h
2025-10-05 17:36:14,315 INFO   g

In [ ]:
df.shape

In [ ]:

df_pred = df.tail(1000)
df_pred.to_csv("/content/PatchTST/dataset/stock_pred.csv", index=False)

In [ ]:
#!/bin/bash

!python3 -u PatchTST/run.py \
  --root_path PatchTST/dataset/ \
  --data_path stock_pred.csv \
  --model PatchTST \
  --features S \
  --target close \
  --seq_len 96 \
  --label_len 48 \
  --pred_len 1 \
  --e_layers 3 \
  --d_model 128 \
  --d_ff 256 \
  --n_heads 16 \
  --dropout 0.1 \
  --fc_dropout 0.1 \
  --head_dropout 0.0 \
  --patch_len 16 \
  --stride 8 \
  --batch_size 32 \
  --learning_rate 1e-4 \
  --train_epochs 10 \
  --use_vmd \
  --num_imfs 3 \
  --do_predict \
  --checkpoints PatchTST/checkpoints/ \
  --des "experiment"


In [ ]:
preds_imfs = np.load('/content/results/experiment_PatchTST_ftS_sl96_ll48_pl1_dm128_nh16_el3_df256_VMD3_ASWL_0/pred/pred_imfs.npy')
trues_imfs = np.load('/content/results/experiment_PatchTST_ftS_sl96_ll48_pl1_dm128_nh16_el3_df256_VMD3_ASWL_0/pred/true_imfs.npy')


In [ ]:
trues_imfs.shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# preds_imfs: shape (N, K)   where K = num_imfs
# trues_imfs: shape (N, K)

num_imfs = preds_imfs.shape[1]

plt.figure(figsize=(15, num_imfs * 3))

for i in range(num_imfs):
    plt.subplot(num_imfs, 1, i+1)
    plt.plot(trues_imfs[-100:-10, i], label=f"True IMF {i+1}", marker="o", markersize=3)
    plt.plot(preds_imfs[-100:-10, i], label=f"Pred IMF {i+1}", marker="x", markersize=3)
    plt.title(f"IMF {i+1}: Prediction vs Ground Truth")
    plt.xlabel("Time step")
    plt.ylabel("Normalized Value")
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load arrays
preds = np.load('/content/results/experiment_PatchTST_ftS_sl96_ll48_pl1_dm128_nh16_el3_df256_VMD3_ASWL_0/pred/pred_final.npy')
trues = np.load('/content/results/experiment_PatchTST_ftS_sl96_ll48_pl1_dm128_nh16_el3_df256_VMD3_ASWL_0/pred/true_final.npy')

# preds and trues are 1D arrays
print("Preds shape:", preds.shape)
print("Trues shape:", trues.shape)

# Take first 20 points for visualization
pred_20 = preds[-100:]
true_20 = trues[-100:]

plt.figure(figsize=(12, 6))
plt.plot(true_20, label="Ground Truth", marker="o")
plt.plot(pred_20, label="Prediction", marker="x")
plt.title("Predictions vs Ground Truth (First 20 points)")
plt.xlabel("Time step")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.show()



In [ ]:
df_pred = df.head(97)
df_pred.to_csv("/content/PatchTST/dataset/stock_pred.csv", index=False)